# 1  最原始的写法

In [1]:
file = open("my_file.txt")
contents = file.read()
print(contents)
file.close()

New data.


以下是逐行分析：

## 1.1 Python built-in method `open()`

In [2]:
file = open("my_file.txt")

先打一个比方，来建立直觉。然后再说真正的技术细节。

> 你的电脑硬盘上躺着一个文件 `my_file.txt`，它就像图书馆书架上的一本书。这本书安安静静地躺在那儿， Python 程序**摸不到它**——Python 活在内存里，硬盘上的东西它不能直接伸手去翻。它必须跟操作系统说：「帮我把那本书拿下来、打开、摆到桌上。」操作系统答应了，就在你的程序和那个硬盘文件之间搭了一条通道。
> 
> `open()` 做的就是这件事——**向操作系统申请一条通往硬盘文件的通道**。


`open("my_file.txt")` 返回的是一个 **file object**，具体来说是 `io.TextIOWrapper` 这个类的实例。

In [3]:
# 验证一下

print(type(file))

<class '_io.TextIOWrapper'>


这个对象**不是文件本身的内容**。它不是那个 `"Hello this is my file."` 字符串。它是那条**通道**——一个活的、有状态的连接。它记住了：我连的是哪个文件、我现在读到哪个位置了（有一个内部的"光标"）、我是只读模式还是写入模式。

`file` 这个变量名，就指向了这个 `TextIOWrapper` 实例。

## 1.2 `.read()`方法

### 1.2.1 `.read()`的基本内容

In [4]:
contents = file.read()

`.read()` 是 `TextIOWrapper` 对象身上的一个方法。

它做的事情是：**沿着那条通道，把文件里的内容从硬盘搬进内存，以字符串的形式返回。**

In [5]:
print(contents)

print(type(contents))

New data.
<class 'str'>


所以 `contents` 指向的确实就是一个 `str`。如果文件里写的是 `Hello this is my file.`，那 `contents` 就指向字符串 `"Hello this is my file."`。

### 1.2.2 光标的概念

还有一个细节：读完之后，那个内部"光标"就移到了文件末尾。

如果紧接着再调一次 `file.read()`，会得到空字符串 `""`。

（下面打印的是空白，但这是因为不显示`""`这对引号而已，只显示了字符串里实际的内容，实际的内容正好就是空白）

In [6]:
contents2 = file.read()
print(contents2)
print(type(contents2))


<class 'str'>


不是文件里面的东西被删了，是光标已经在末尾了，没有新东西可读了。想再读一遍，得用 `file.seek(0)` 把光标挪回开头。

In [7]:
file.seek(0)
contents3 = file.read()
print(contents3)
print(type(contents3))

New data.
<class 'str'>


## 1.3 `.close()`方法

`close()`方法负责**拆掉那条通道**。告诉操作系统：我用完了，你可以回收资源了。

In [8]:
file.close()

#### 为什么必须 close？

##### (1) 因为操作系统能同时维持的通道数量是有限的
每打开一个文件，操作系统就要分配一个叫 **file descriptor** 的东西（可以理解为"通道编号"），全系统一共就那么多编号。

你不 close，编号就一直被占着。写一个小脚本无所谓，但如果你的程序循环打开几千个文件都不关，编号用光了，操作系统就会拒绝你：`OSError: Too many open files`。

##### (2) `.close()` 可以把缓冲区里的东西强行写入硬盘
对于写入模式 `file.write("xxx", "w")` ，数据不一定立刻写进硬盘——它可能先存在内存里的一个缓冲区里，等攒够了一批再一起写入，等攒够了一批再一起写入（这样效率更高）。

`.close()` 会强制把缓冲区里的东西全刷进硬盘，然后再断开通道。如果不 close 就让程序结束了，缓冲区里的数据可能就丢了。






现在补上了，整条链就通了：`open()` 建通道 → 返回 file object → `.read()` 沿通道搬数据进内存变成 str → `.close()` 拆通道还资源 → `with` 帮你自动拆。